In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import yaml
import zarr
from tqdm import tqdm
import pandas as pd
import polars as pl
import numpy as np
from plotnine import *

from anngeno import AnnGeno
from scripts import get_burdens, get_burdens_faster, get_correlations

## Compute burdens

In [ ]:
# a = pl.read_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/Beltran_nature_2025/beltran_genebass_genes_associations.parquet')
a = pl.read_parquet('/s/project/deeprvat/ukb_gym/rap_data/small_gene_assocs_with_region_id.pq')
a.shape

In [ ]:
config_path = './config_rap.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

maf = config.get("maf_upper_bound")

associations_df_path = '/s/project/deeprvat/ukb_gym/rap_data/small_gene_assocs_with_region_id.pq'
associations_df = pl.read_parquet(associations_df_path)
genes = associations_df['region'].unique()
genes_list = genes.to_list()

print("Loading AnnGeno file")
anngeno_file = config.get("anngeno_file")
ag = AnnGeno(filename=anngeno_file, filemode="r")

print(f"Filtering for variants with MAF < {maf}")
variants_to_keep_df = ag.annotations.filter(
    (pl.col('MAF') < maf) & 
    (pl.col('region').is_in(genes_list))
)

print(f"Filtering for SNPs")
variants_to_keep_df = ag.annotations.filter(
    (pl.col("ref").str.len_chars() == 1) &
    (pl.col("alt").str.len_chars() == 1)
)

variants_to_keep = set(variants_to_keep_df["id"])
ag.subset_variants(variants_to_keep)

variants_to_keep_df.shape

In [ ]:
a = pl.read_parquet('/s/project/deeprvat/ukb_gym/rap_data/small_gene_assocs_with_region_id.pq')
a

In [ ]:
variants_to_keep_df.filter(pl.col('region')==2256).shape

In [ ]:
a.filter(pl.col('region')==2256).shape

In [ ]:
ag.annotations.columns

In [ ]:
anngeno_obj = ag 
annotation_list = ['loftee_hc', 'alphamissense', 'CADD_raw', 'sift_score', 'Consequence_missense_variant', 'Consequence_protein_altering_variant']

gene_burdens_sum = []
gene_burdens_max = []
for gene_num in tqdm(genes_list):
    gene = anngeno_obj.get_region(gene_num)
    geno = gene["genotypes"]

    subset_anno = gene["annotations"][["id", *annotation_list]]

    no_variant_mask = geno.sum(axis = 1) == 0
    var_scores = subset_anno[annotation_list].fill_nan(0).to_numpy().astype(np.float32)

    # Calculate sum burden directly
    gis_sum = np.dot(geno, var_scores)
    gis_sum[no_variant_mask, :] = np.nan

    # For max calculation
    gis_max = []
    for a in tqdm(range(var_scores.shape[1])):
        gis_max.append(np.max(geno*var_scores[:, a], axis=1))
    gis_max = np.stack(gis_max, axis=1)
    gis_max[no_variant_mask, :] = np.nan

    gene_burdens_sum.append(gis_sum)
    gene_burdens_max.append(gis_max)

gene_burdens_sum_df = np.stack(gene_burdens_sum, axis=1)
gene_burdens_max_df = np.stack(gene_burdens_max, axis=1)


In [ ]:
gene_burdens_sum_df

## Save burdens

In [ ]:
zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/rap_wgs_55genes.zarr'
overwrite = True
anno_chunk_size = 1
max_burden = True
all_annotation_list = annotation_list
sample_id_arr = ag.samples 
gene_id_list = genes_list

root = zarr.group(zarr_burdens_path)
root.create_array(
    "sum_burdens",
    shape=gene_burdens_sum_df.shape,
    dtype=gene_burdens_sum_df.dtype,
    chunks=(gene_burdens_sum_df.shape[0], gene_burdens_sum_df.shape[1], anno_chunk_size),
    overwrite=overwrite,
)[:] = gene_burdens_sum_df

if max_burden:
    root.create_array(
        "max_burdens",
        shape=gene_burdens_max_df.shape,
        dtype=gene_burdens_max_df.dtype,
        chunks=(gene_burdens_max_df.shape[0], gene_burdens_max_df.shape[1], anno_chunk_size),
        overwrite=overwrite,
    )[:] = gene_burdens_max_df

root.create_array("samples", shape=sample_id_arr.shape, dtype="str", overwrite=overwrite)[:] = sample_id_arr
root.create_array("genes", shape=len(gene_id_list), dtype="str", overwrite=overwrite)[:] = gene_id_list
root.create_array("annotations", shape=len(all_annotation_list), dtype="str", overwrite=overwrite)[:] = all_annotation_list
print("Created new zarr array 'sum_burdens', 'samples', 'genes', and 'annotations'.")

In [ ]:
zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/rap_wgs_55genes.zarr'

zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
sample_list = zarr_group["samples"][:]
gene_list = zarr_group["genes"][:]
annotation_list = zarr_group["annotations"][:]

anno = 'loftee_hc'
anno_idx = np.where(annotation_list == anno)[0][0]
sum_burdens = zarr_group["sum_burdens"][:, :, anno_idx]
sum_burdens

In [ ]:
gene_list

## Compute correlations

In [ ]:
import statsmodels.api as sm

# Get covariate corrected phenotypes
def cov_prs_correction(all_df, phenotypes, covariates, prs_pheno_map):
    # Initialize an empty DataFrame to store residuals
    # all_df.set_index('sample', inplace=True)
    cov_prs_corrected_phenos = pd.DataFrame(index=all_df.index)  # Index is the sample ID

    # Perform linear regression for each phenotype
    for pheno in tqdm(phenotypes):
        # Drop NaN values for the current phenotype
        combined_df = all_df[[pheno] + covariates + [prs_pheno_map[pheno]]].dropna()
        y = combined_df[pheno]
        X = combined_df.drop(columns=[pheno])
        X = sm.add_constant(X)  # Add a constant term for the intercept

        # Fit the model
        model = sm.OLS(y, X).fit()

        # Save residuals
        residuals = pd.Series(model.resid, index=combined_df.index, name=pheno)
        cov_prs_corrected_phenos = pd.concat(
            [cov_prs_corrected_phenos, residuals], axis=1
        )

    # Reset the index for the resulting DataFrame
    cov_prs_corrected_phenos.reset_index(inplace=True)
    # cov_prs_corrected_phenos.columns = ['sample'] + [f"{pheno}_cov_prs_corrected" for pheno in phenotypes]
    return cov_prs_corrected_phenos


def pheno_burden_correlation(assoc_df, gt_df, annotation, correlation_type):
    rank_corr_list = []
    for trait in assoc_df.phenotype.unique():
        assoc_df['region'] = assoc_df['region'].astype(int)
        gene_list = list(assoc_df.query("phenotype == @trait")['region'].astype(str))
        pheno = trait.replace(" ", "_")
        for gene in gene_list:
            try:
                correlation = (
                    gt_df[[gene, pheno]].dropna().corr(method=correlation_type).iloc[0, 1]
                )
            except ValueError:
                print("Wrong correlation type specified. Reverting to spearman")
                correlation = (
                    gt_df[[gene, pheno]].dropna().corr(method="spearman").iloc[0, 1]
                )
            except Exception as e:
                print(
                    f"Cannot compute correlation for {annotation}, {pheno}, {gene}. Error: {e}"
                )
                correlation = np.nan
                correlation_non_zero = np.nan

            rank_corr_list.append(
                pd.DataFrame(
                    {
                        "annotation": annotation,
                        "phenotype": trait,
                        "gene": gene,
                        "correlation": correlation,
                    },
                    index=[0],
                )
            )
    return pd.concat(rank_corr_list)

In [ ]:
config_path = './config_rap.yaml'
associations_df_path = '/s/project/deeprvat/ukb_gym/rap_data/small_gene_assocs_with_region_id.pq'

with open(config_path) as f:
    config = yaml.safe_load(f)

anngeno_file = config.get("anngeno_file")
phenotypes = config.get("phenotypes_for_testing")
covs = config.get("covariates")
prs_pheno_map_file = config.get("prs_pheno_map_file")
prs_file = config.get("prs_file")

cov_pheno_df = pd.read_parquet(
    f"{anngeno_file}/phenotypes.parquet", columns=["sample"] + covs + phenotypes
).set_index("sample")
prs_df = pd.read_parquet(prs_file)
prs_df = prs_df[prs_df.index.isin(cov_pheno_df.index)]
prs_pheno_map = pd.read_csv(prs_pheno_map_file)
prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
pheno_corrected_df = cov_prs_correction(all_df, phenotypes, covs, prs_pheno_map)

# if genes_to_keep is not None:
#     assoc_df = assoc_df[assoc_df.gene.isin(genes_to_keep)]

In [ ]:
zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/rap_wgs_55genes.zarr'
correlation_type = 'spearman'
max_burden = True

zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
sample_list = zarr_group["samples"][:]
gene_list = zarr_group["genes"][:]
annotation_list = zarr_group["annotations"][:]

assoc_df = pd.read_parquet(associations_df_path)
assoc_df

In [ ]:
rho_df_sum_list = []
rho_df_max_list = []
print(f"Starting correlation computation for {len(annotation_list)} annotations")
print(annotation_list)
for anno in tqdm(annotation_list):
    anno_idx = np.where(annotation_list == anno)[0][0]
    sum_burdens = zarr_group["sum_burdens"][:, :, anno_idx]
    gt_df_sum = pd.DataFrame(sum_burdens, index=sample_list, columns=gene_list).merge(pheno_corrected_df, left_index=True, right_on="sample")
    rho_df_sum_list.append(pheno_burden_correlation(assoc_df, gt_df_sum, anno, correlation_type))
    
    if max_burden:
        max_burdens_zarr = zarr_group["max_burdens"][:, :, anno_idx]
        gt_df_max = pd.DataFrame(max_burdens_zarr, index=sample_list, columns=gene_list).merge(pheno_corrected_df, left_index=True, right_on="sample")
        rho_df_max_list.append(pheno_burden_correlation(assoc_df, gt_df_max, anno, correlation_type))

rho_df_sum = pd.concat(rho_df_sum_list)
rho_df_sum["aggregation"] = "sum"

if max_burden:
    rho_df_max = pd.concat(rho_df_max_list)
    rho_df_max["aggregation"] = "max"
    rho_df = pd.concat([rho_df_sum, rho_df_max])


In [ ]:
rho_df[rho_df['correlation'] == rho_df['correlation'].max()]

In [ ]:
rho_df[~rho_df['correlation'].isna()]['annotation'].value_counts()

## Plot trends

In [ ]:
rank_corr_sum = rho_df[rho_df['aggregation']=='sum'].copy()
rank_corr_sum['abs_correlation'] = np.abs(rank_corr_sum['correlation'])
rank_corr_sum['median_abs_correlation'] = rank_corr_sum.groupby(['annotation', 'aggregation'])['abs_correlation'].transform('median')
rank_corr_sum = rank_corr_sum.sort_values('median_abs_correlation', ascending=False)
rank_corr_sum['annotation'] = pd.Categorical(rank_corr_sum['annotation'], categories=rank_corr_sum['annotation'].unique(), ordered=True)

# annotation_list = ['alphamissense', 'PrimateAI_score', 'sift_score', 'Consequence_missense_variant', 'Consequence_protein_altering_variant']
# rank_corr_sum = rank_corr_sum[~rank_corr_sum['annotation'].isin(annotation_list)]
(
    ggplot(rank_corr_sum, aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    ylab('|rank correlation|') +
    # facet_wrap('~aggregation', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(7, 6),
    )
)

In [ ]:
rank_corr_max = rho_df[rho_df['aggregation']=='max'].copy()
rank_corr_max['abs_correlation'] = np.abs(rank_corr_max['correlation'])
rank_corr_max['median_abs_correlation'] = rank_corr_max.groupby(['annotation', 'aggregation'])['abs_correlation'].transform('median')
rank_corr_max = rank_corr_max.sort_values('median_abs_correlation', ascending=False)
rank_corr_max['annotation'] = pd.Categorical(rank_corr_max['annotation'], categories=rank_corr_max['annotation'].unique(), ordered=True)


(
    ggplot(rank_corr_max, aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    # facet_wrap('~aggregation', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(7, 6),
    )
)

In [ ]:
rank_corr_max

# Phenotype vs GIS plot

In [ ]:
import zarr
from scripts import get_correlation

def pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_burdens_path):

    with open(config_path) as f:
        config = yaml.safe_load(f)

    anngeno_file = config.get('anngeno_file')
    # annotation_list = config.get('rare_variant_annotations')
    # phenotypes = config.get('phenotypes_for_association_testing')
    covs = config.get('covariates')
    prs_pheno_map_file = config.get('prs_pheno_map_file')
    prs_file = config.get('prs_file')

    cov_pheno_df = pd.read_parquet(f"{anngeno_file}/phenotypes.parquet", columns=['sample'] + covs + [phenotype]).set_index('sample')
    prs_df = pd.read_parquet(prs_file)
    prs_df = prs_df[prs_df.index.isin(cov_pheno_df.index)]
    prs_pheno_map = pd.read_csv(prs_pheno_map_file)
    prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
    all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
    pheno_corrected_df = get_correlation.cov_prs_correction(all_df, [phenotype], covs, prs_pheno_map)

    zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
    sample_list = zarr_group['samples'][:]
    gene_list = zarr_group["genes"][:]
    annotation_list = zarr_group["annotations"][:]

    gene_idx = np.where(gene_list == gene_num)[0][0]
    anno_idx = np.where(annotation_list == annotation)[0][0]
    burden_type = burden_type.lower()
    if burden_type == "max":
        burdens = zarr_group["max_burdens"][:, gene_idx, anno_idx]
    elif burden_type == "sum":
        burdens = zarr_group["sum_burdens"][:, gene_idx, anno_idx]
    else:
        raise ValueError("burden_type must be either 'max' or 'sum'")

    burden_df = pd.DataFrame(burdens, index=sample_list, columns=[gene_num]).merge(pheno_corrected_df, left_index=True, right_on='sample')
    burden_df = burden_df.dropna()
    gis_mode = burden_df[gene_num].mode()[0]
    burden_df_non_zero = burden_df[burden_df[gene_num]!=gis_mode]
    correlation = burden_df[[gene_num, phenotype]].dropna().corr(method='spearman').iloc[0, 1]

    return burden_df, burden_df_non_zero, correlation

In [ ]:
phenotype = 'Mean_platelet_thrombocyte_volume' #'Urate'
gene_num = '23103' #'9231'
	
annotation = 'scaled_fitness' #'pangolin_score'
burden_type = 'sum'

plt_df, plt_df_nz, c = pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_burdens_path)

(
    ggplot(plt_df, aes(x=gene_num, y=phenotype)) +
    geom_point(alpha=0.5) +
    geom_smooth(method='lm', se=False) +
    labs(x='LDLR',
         y=phenotype) +
    theme_bw() +
    labs(title = f"{annotation} - {round(c, 4)}")

)